# 🤗 LeRobot Local Inference — Run HuggingFace Models Directly

Instead of connecting to an inference server, `LerobotLocalPolicy` loads
HuggingFace VLA models right on your GPU. Any model that LeRobot supports
works automatically: **ACT, Pi0, SmolVLA, Diffusion, XVLA**, and more.

**Requirements:** `pip install "strands-robots[lerobot]"` + a GPU

> 💡 This notebook explains the concepts and architecture. The GPU-requiring
> cells are shown as code blocks in markdown so you can follow along without hardware.

## How It Works

```
Your robot observation (dict of images + joint values)
    │
    ▼
ProcessorBridge.preprocess()     ← normalize, resize, tokenize
    │
    ▼
LeRobot policy.select_action()   ← the actual neural network
    │
    ▼
ProcessorBridge.postprocess()    ← unnormalize, convert to robot format
    │
    ▼
list[dict] — one dict per timestep, keys are your joint names
```

The key insight: **you never touch tensors directly**. The policy handles all the
conversion between robot-friendly dicts and model-friendly tensors.

## Loading a Model

```python
from strands_robots import create_policy

# The factory auto-detects "lerobot_local" from the HuggingFace org prefix
policy = create_policy("lerobot/act_aloha_sim_transfer_cube_human")

# Or be explicit about the provider
policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path="lerobot/act_aloha_sim_transfer_cube_human",
    device="cuda",
    actions_per_step=1,     # how many timesteps to return per call
    use_processor=True,      # load the model's pre/post processing pipeline
)

# Map tensor indices to your robot's actual joint names
policy.set_robot_state_keys([
    "waist", "shoulder", "elbow", "forearm_roll",
    "wrist_angle", "wrist_rotate", "gripper",
])

# If you skip set_robot_state_keys(), it auto-generates: joint_0, joint_1, ...
```

## How Models Are Found

When you pass a model name, the library needs to find the right LeRobot class.
This is trickier than it sounds — LeRobot's API changed significantly between
versions, and different models use different class hierarchies.

The resolution chain:
1. **draccus config registry** (LeRobot 0.5+) — the modern way
2. **Manual config.json parsing** — reads the `"type"` field from HuggingFace
3. **Direct module import** — `lerobot.policies.act.modeling_act`
4. **Legacy factory** — `lerobot.policies.factory.get_policy_class` (v0.4)

A clever hack: a lightweight stub is injected for `lerobot.policies` to avoid
importing its heavy `__init__.py` (which pulls in groot → flash_attn and can crash).

In [ ]:
# You can test resolution without downloading any model weights:
try:
    from strands_robots.policies.lerobot_local.resolution import resolve_policy_class_by_name
    ACT = resolve_policy_class_by_name("act")
    print(f"✅ ACT class resolved: {ACT.__name__}")
except ImportError as e:
    print(f"LeRobot not installed: {e}")

## The Processor Bridge

Many models ship with `preprocessor.json` and `postprocessor.json` — pipeline
configs for normalizing observations and unnormalizing actions. The `ProcessorBridge`
loads these automatically:

In [ ]:
from strands_robots.policies.lerobot_local.processor import (
    ProcessorBridge, PREPROCESSOR_CONFIG, POSTPROCESSOR_CONFIG,
)

# Without a model, it's a passthrough (no-op)
bridge = ProcessorBridge()
print(f"Active: {bridge.is_active}")          # False
print(f"Has preprocessor: {bridge.has_preprocessor}")  # False
print(f"Has postprocessor: {bridge.has_postprocessor}") # False
print(f"Config files: {PREPROCESSOR_CONFIG}, {POSTPROCESSOR_CONFIG}")

```python
# With a real model, it loads the pipeline:
bridge = ProcessorBridge.from_pretrained(
    "lerobot/act_aloha_sim",
    device="cuda",
    overrides={"image_size": [224, 224]},
)

processed_obs = bridge.preprocess(raw_obs, instruction="pick up cube")
final_action = bridge.postprocess(raw_action_tensor)
bridge.reset()  # clear stateful step state between episodes
```

Under the hood, `preprocess()` builds a full LeRobot `Transition` with
complementary data — this is how VLA tokenizer steps access the `"task"` instruction.

## Real-Time Chunking (RTC)

Flow-matching policies like **Pi0** and **SmolVLA** support RTC — blending action
chunks across inference calls to compensate for GPU compute latency.

The idea: while the GPU is computing the next chunk, the robot is still executing
the previous chunk. RTC tracks this latency and adjusts the blend.

```python
policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path="lerobot/pi0_aloha",
    rtc_enabled=True,              # auto-detected from model config if None
    rtc_execution_horizon=10,      # timesteps from prefix for guidance
    rtc_max_guidance_weight=10.0,  # max correction weight
)
```

> ⚠️ RTC only works with flow-matching policies that have `rtc_config` in their model config.
> ACT and Diffusion policies do NOT support it — they use `select_action()` with
> LeRobot's built-in temporal ensemble instead.

## Episode Reset — Don't Skip This!

```python
for episode in range(n_episodes):
    policy.reset()  # Clears:
                    #   - LeRobot's internal action queue
                    #   - Temporal ensemble buffers
                    #   - RTC prev_chunk and latency history
                    #   - ProcessorBridge stateful step state
    obs = env.reset()
    for step in range(max_steps):
        actions = policy.get_actions_sync(obs, instruction)
        obs = env.step(actions[0])
```

Without `reset()`, stale actions from the previous episode leak into the next one.